# [v3] Train GOOD/BAD bằng 1 cam hông 90° (RULA nhóm B → Transformer) + đo lệch vai bằng công thức (cam laptop)

**TRAIN (1 cam hông 90°)**
Video 90° → MediaPipe Pose (~10 fps) → lọc nhiễu → góc cổ / thân / cúi đầu → hiệu chỉnh theo người (clip CALIBRATION)
→ **RULA nhóm B (teacher)** → GOOD/BAD + lọc khung sát ngưỡng → cửa sổ 2 s → **Transformer (student)** → đánh giá (LOSO) → **ONNX**

**INFERENCE (2 cam)**
- Cam hông 90° → MediaPipe → Student ONNX → GOOD/BAD (cổ, thân)
- Cam laptop chính diện → MediaPipe → **công thức lệch vai (không train)** → SHOULDER_TILT
- Gộp theo thời gian + median filter + debounce → cảnh báo

**Thay đổi so với v2**
- Teacher đổi từ RULA rút gọn có tay (Table A/C) sang **RULA nhóm B** (cổ – thân – chân) + luật bổ sung ngả sau > 15° là BAD.
- Bỏ đặc trưng tay / khuỷu / cổ tay. Đặc trưng còn: cổ, thân, cúi đầu + visibility.
- Cam chạy thật đặt **90° như lúc train** → bỏ augmentation mô phỏng cam 70°, chỉ giữ nhiễu nhỏ.
- Lệch vai **không train**: công thức trên cam laptop, kiểm tra trên clip SHOULDER_TILT (Phần 7).
- Có thêm baseline Random Forest để so với Transformer.
- Script realtime chạy **2 cam**.

**Cách chạy**
1. Tạo Kaggle Dataset từ thư mục `Video_TrainAI_KhoaLuan` (có P01, P02, P03). Nén **ZIP** nếu cần (không RAR).
2. **Add Input** dataset đó, **Settings → Internet: On** (tải mediapipe + model pose). GPU không bắt buộc.
3. Sửa `DATA_DIR` ở Phần 1 → **Run All**.
4. Tải `posture_model.zip` ở Output → giải nén → `pip install -r requirements.txt` → `python realtime_posture.py --side 1 --front 0`.

## Phần 0 — Cài đặt

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mediapipe', 'onnx', 'onnxruntime'], capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-1500:])
import mediapipe as mp, torch, numpy as np, pandas as pd, cv2, onnx
print('mediapipe', mp.__version__, '| torch', torch.__version__, '| GPU:', torch.cuda.is_available())

## Phần 1 — Cấu hình

In [ ]:
import os, glob, re
from pathlib import Path

# ---- SỬA: thư mục dataset (chứa P01, P02, P03 — tìm cả thư mục con) ----
DATA_DIR = '/kaggle/input/datasets/vphcvit12a1/datakl/Video_TrainAI_KhoaLuan'

WORK = '/kaggle/working'
LM_DIR = f'{WORK}/lm'; OUT_DIR = f'{WORK}/posture_model'
os.makedirs(LM_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
os.chdir(WORK)

allv = sorted(str(p) for p in Path(DATA_DIR).rglob('*') if p.suffix.lower() in ('.mp4', '.mov'))
allv = [v for v in allv if '(1)' not in v and re.match(r'P\d+_S\d+_', Path(v).stem)]   # bỏ file trùng
side_videos = [v for v in allv if 'SHOULDER_TILT' not in v]     # cam hông 90° -> train
front_videos = [v for v in allv if 'SHOULDER_TILT' in v]        # cam trước -> chỉ kiểm tra công thức lệch vai
if not side_videos:
    print(os.listdir(DATA_DIR)); raise FileNotFoundError('Không thấy video — sửa DATA_DIR')
print(len(side_videos), 'video cam hông; người:', sorted({Path(v).stem[:3] for v in side_videos}))
print(len(front_videos), 'video cam trước (SHOULDER_TILT):', [Path(v).stem for v in front_videos])

## Phần 2 — Module đặc trưng + RULA nhóm B + công thức lệch vai (dùng chung cho train và realtime)

In [ ]:
%%writefile /kaggle/working/posture_features.py
"""
Đặc trưng tư thế + RULA nhóm B + công thức lệch vai — DÙNG CHUNG cho lúc train và lúc chạy realtime.

- Cam hông 90° (train + chạy thật): góc cổ / thân / cúi đầu -> RULA nhóm B (teacher) -> Transformer (student).
- Cam laptop chính diện (chỉ lúc chạy thật): công thức lệch vai, KHÔNG train.

Đầu vào mỗi khung: 33 landmark MediaPipe Pose 2D (toạ độ pixel x, y, visibility),
world landmark 3D chỉ dùng để chọn phía cơ thể gần camera.
"""
import math
import numpy as np

# chỉ số MediaPipe Pose: (trái, phải)
IDX = {"ear": (7, 8), "eye": (2, 5), "shoulder": (11, 12), "hip": (23, 24)}
NOSE = 0

FEATURES = ["neck", "trunk", "head_pitch", "vis_ear", "vis_shoulder", "vis_hip"]
CALIBRATED = ["neck", "trunk", "head_pitch"]      # trừ giá trị lúc ngồi chuẩn của từng người
ANGLE_SCALE = 30.0                                 # chia góc cho 30° trước khi vào model

# ================================================================ CAM HÔNG 90°
def choose_side(lm_seq, world_seq=None):
    """0 = trái, 1 = phải. Ưu tiên world z (nhỏ = gần camera), dự phòng bằng visibility."""
    if world_seq is not None and np.isfinite(world_seq).any():
        zl = np.nanmedian(world_seq[:, 11, 2] + world_seq[:, 23, 2])
        zr = np.nanmedian(world_seq[:, 12, 2] + world_seq[:, 24, 2])
        if np.isfinite(zl) and np.isfinite(zr):
            return 0 if zl < zr else 1
    vl = np.nanmean(lm_seq[:, [7, 11, 23], 2]); vr = np.nanmean(lm_seq[:, [8, 12, 24], 2])
    return 0 if vl >= vr else 1


def facing_sign(lm_seq, side):
    """+1 nếu người quay mặt sang phải khung hình, -1 nếu sang trái (mũi so với tai)."""
    d = np.nanmedian(lm_seq[:, NOSE, 0] - lm_seq[:, IDX["ear"][side], 0])
    return 1.0 if (np.isfinite(d) and d > 0) else -1.0


def _ang(v, f):
    """Góc (độ) của vector v so với phương thẳng đứng hướng LÊN, dương khi nghiêng về phía mặt."""
    return math.degrees(math.atan2(v[0] * f, -v[1]))


def frame_angles(p, side, f):
    """p: (33,3) pixel x,y,vis. Trả dict góc (độ) + visibility, None nếu thiếu tai/vai/hông."""
    g = lambda k: p[IDX[k][side], :2]
    ear, eye, sh, hip = g("ear"), g("eye"), g("shoulder"), g("hip")
    if not np.isfinite(np.concatenate([ear, sh, hip])).all():
        return None
    trunk = _ang(sh - hip, f)                       # thân so với phương đứng (+ = gập ra trước, − = ngả sau)
    neck = _ang(ear - sh, f) - trunk                # cổ so với đường thân (+ = gập)
    # cúi đầu: đường tai -> mắt so với phương ngang (+ = mắt thấp hơn tai = cúi)
    head_pitch = math.degrees(math.atan2(eye[1] - ear[1], (eye[0] - ear[0]) * f)) if np.isfinite(eye).all() else np.nan
    vis = lambda k: float(p[IDX[k][side], 2])
    return dict(neck=neck, trunk=trunk, head_pitch=head_pitch,
                vis_ear=vis("ear"), vis_shoulder=vis("shoulder"), vis_hip=vis("hip"))


def sequence_angles(lm_seq, side, f):
    """(T,33,3) -> (T, len(FEATURES)) góc thô (chưa hiệu chỉnh), NaN nếu thiếu."""
    out = np.full((len(lm_seq), len(FEATURES)), np.nan, np.float32)
    for t, p in enumerate(lm_seq):
        if not np.isfinite(p).any():
            continue
        a = frame_angles(p, side, f)
        if a is not None:
            out[t] = [a[k] for k in FEATURES]
    return out


def calibration_baseline(raw):
    """raw: (T,F) góc thô lúc ngồi chuẩn -> dict baseline cho các góc cần hiệu chỉnh."""
    return {k: float(np.nanmedian(raw[:, FEATURES.index(k)])) for k in CALIBRATED}


def apply_calibration(raw, base):
    x = raw.copy()
    for k in CALIBRATED:
        x[:, FEATURES.index(k)] -= base[k]
    return x


def to_model_input(cal):
    """Góc đã hiệu chỉnh -> mảng chuẩn hoá cho model (NaN -> 0, visibility giữ nguyên 0..1)."""
    x = cal.copy()
    for i, k in enumerate(FEATURES):
        if not k.startswith("vis_"):
            x[:, i] = x[:, i] / ANGLE_SCALE
    return np.nan_to_num(x, nan=0.0).astype(np.float32)

# ================================================================ RULA NHÓM B (teacher)
# McAtamney & Corlett (1993). Chỉ Cổ – Thân – Chân; bỏ nhóm A, điểm cơ, điểm lực.
# Side cam không thấy xoay / nghiêng sang bên -> các cờ điều chỉnh = 0. Chân mặc định = 1.
TABLE_B = {  # neck -> [T1L1, T1L2, T2L1, T2L2, ..., T6L1, T6L2]
    1: [1, 3, 2, 3, 3, 4, 5, 5, 6, 6, 7, 7], 2: [2, 3, 2, 3, 4, 5, 5, 5, 6, 7, 7, 7],
    3: [3, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 7], 4: [5, 5, 5, 6, 6, 7, 7, 7, 7, 7, 8, 8],
    5: [7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8], 6: [8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9],
}
BAD_SCORE = 3          # điểm nhóm B >= 3 -> BAD
NEUTRAL_TOL = 5.0      # thân: <= 5° so với lúc hiệu chỉnh vẫn coi là thẳng (1 điểm)
NECK_EXT_TOL = 15.0    # cổ chỉ tính "ngửa" (4 điểm) khi ngửa hơn lúc hiệu chỉnh > 15°
LEAN_BACK_TOL = 15.0   # luật bổ sung ngoài RULA: thân ngả sau > 15° -> BAD (LEAN_BACKWARD)
HEAD_FROM_PITCH = True # góc cổ = max(tai–vai so với thân, cúi mặt tai->mắt): bắt kiểu gù mà cúi mặt


def score_neck(a):
    if a < -NECK_EXT_TOL: return 4
    if a <= 10:           return 1
    if a <= 20:           return 2
    return 3


def score_trunk(a):
    if a <= NEUTRAL_TOL: return 1          # thẳng, hoặc ngả sau có tựa
    if a <= 20:          return 2
    if a <= 60:          return 3
    return 4


def group_b_score(neck, trunk, legs=1):
    """Góc cổ, thân (độ, đã hiệu chỉnh) -> điểm nhóm B (1..9), NaN nếu thiếu."""
    if not (np.isfinite(neck) and np.isfinite(trunk)):
        return np.nan
    return TABLE_B[score_neck(neck)][(score_trunk(trunk) - 1) * 2 + (legs - 1)]


def b_label(neck, trunk):
    """-> 0 = GOOD, 1 = BAD, NaN nếu thiếu."""
    b = group_b_score(neck, trunk)
    if not np.isfinite(b):
        return np.nan
    return int(b >= BAD_SCORE or trunk < -LEAN_BACK_TOL)


def rula_frame_label(cal_row, margin=2.0):
    """cal_row: 1 khung đặc trưng đã hiệu chỉnh. Trả (nhãn 0/1/NaN, ổn_định?).
    'Ổn định' = nhãn không đổi khi xê dịch cổ/thân ±margin độ (khung sát ngưỡng sẽ không ổn định)."""
    n, t, hp = (cal_row[FEATURES.index(k)] for k in ("neck", "trunk", "head_pitch"))
    if HEAD_FROM_PITCH and np.isfinite(hp) and np.isfinite(n):
        n = max(n, hp)
    lab = b_label(n, t)
    if not np.isfinite(lab):
        return np.nan, False
    stable = all(b_label(n + dn, t + dt) == lab for dn in (-margin, margin) for dt in (-margin, margin))
    return int(lab), stable

# ================================================================ CAM LAPTOP: LỆCH VAI (không train)
SHOULDER_TILT_ON = 3.5     # |lệch vai so với lúc hiệu chỉnh| > 3.5° -> lệch (clip P01 nghiêng chủ ý chỉ ~4–5°)
SHOULDER_TILT_OFF = 2.5    # < 2.5° mới coi là hết lệch (trễ, tránh bật/tắt liên tục); ngồi thẳng dao động ~±2°
MIN_SHOULDER_VIS = 0.5     # vai bị che / ra khỏi khung -> bỏ khung


def shoulder_tilt(p):
    """p: (33,3) pixel x,y,vis từ cam CHÍNH DIỆN. Góc (độ) của đường nối 2 vai so với phương ngang.
    Đường vai lấy từ vai bên trái ảnh -> vai bên phải ảnh, nên ảnh lật gương (selfie) vẫn đúng độ lớn.
    Dấu: + = vai bên phải ẢNH thấp hơn. Chỉ dùng |tilt − baseline| để cảnh báo."""
    L, R = p[11], p[12]
    if not (np.isfinite(L).all() and np.isfinite(R).all()) or min(L[2], R[2]) < MIN_SHOULDER_VIS:
        return np.nan
    dx, dy = R[0] - L[0], R[1] - L[1]
    if dx < 0:
        dx, dy = -dx, -dy
    if dx < 1e-6:
        return np.nan
    return math.degrees(math.atan2(dy, dx))


def sequence_shoulder_tilt(lm_seq):
    return np.array([shoulder_tilt(p) if np.isfinite(p).any() else np.nan for p in lm_seq], np.float32)

## Phần 3 — Trích landmark MediaPipe 2D (~10 fps, chạy song song, có cache)

In [ ]:
import urllib.request, time
from concurrent.futures import ProcessPoolExecutor
TASK = f'{WORK}/pose_landmarker_full.task'
if not os.path.exists(TASK):
    urllib.request.urlretrieve('https://storage.googleapis.com/mediapipe-models/pose_landmarker/'
                               'pose_landmarker_full/float16/latest/pose_landmarker_full.task', TASK)
TARGET_FPS = 10

def extract(v):
    import cv2, numpy as np, mediapipe as mp
    from mediapipe.tasks import python as mpt
    from mediapipe.tasks.python import vision
    name = Path(v).stem; out = f'{LM_DIR}/{name}.npz'
    if os.path.exists(out):
        return name, 'cache'
    det = vision.PoseLandmarker.create_from_options(vision.PoseLandmarkerOptions(
        base_options=mpt.BaseOptions(model_asset_path=TASK), running_mode=vision.RunningMode.VIDEO, num_poses=1))
    cap = cv2.VideoCapture(v); fps = cap.get(cv2.CAP_PROP_FPS) or 30; step = max(1, round(fps / TARGET_FPS))
    L, W, T = [], [], []; i = -1; w = h = 0
    while cap.grab():
        i += 1
        if i % step: continue
        ok, fr = cap.retrieve()
        if not ok: break
        h, w = fr.shape[:2]
        sm = cv2.resize(fr, (w // 2, h // 2))
        r = det.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(sm, cv2.COLOR_BGR2RGB)),
                                 int(i * 1000 / fps))
        if r.pose_landmarks:
            L.append([[p.x * w, p.y * h, p.visibility] for p in r.pose_landmarks[0]])
            W.append([[p.x, p.y, p.z] for p in r.pose_world_landmarks[0]])
        else:
            L.append(np.full((33, 3), np.nan)); W.append(np.full((33, 3), np.nan))
        T.append(i / fps)
    cap.release(); det.close()
    np.savez_compressed(out, lm=np.array(L, np.float32).reshape(-1, 33, 3), world=np.array(W, np.float32).reshape(-1, 33, 3),
                        t=np.array(T), fps=fps, wh=np.array([w, h]))
    return name, len(T)

videos = side_videos + front_videos
t0 = time.time()
with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
    for k, (name, n) in enumerate(ex.map(extract, videos), 1):
        print(f'[{k}/{len(videos)}] {name}: {n} khung  ({time.time()-t0:.0f}s)', flush=True)

## Phần 4 — Gán nhãn RULA nhóm B + lọc + cắt cửa sổ (chỉ cam hông)

- Hiệu chỉnh theo từng **người + session + độ phân giải**: trừ góc cổ / thân / cúi đầu trung vị của clip **CALIBRATION** (thiếu thì lấy 10 s đầu các clip GOOD).
- Nhãn khung = **RULA nhóm B** (Table B, chân = 1, không cờ xoay/nghiêng vì side cam không thấy): điểm ≤ 2 → GOOD, ≥ 3 → BAD.
  - Thân ≤ 5° so với lúc hiệu chỉnh vẫn tính là thẳng; cổ chỉ tính *ngửa* khi ngửa hơn lúc hiệu chỉnh > 15°.
  - Góc cổ = max(tai–vai so với thân, cúi mặt tai→mắt).
  - **Luật bổ sung ngoài RULA:** thân ngả sau > 15° → BAD (LEAN_BACKWARD).
- Khung **"sát ngưỡng"** = nhãn đổi khi xê dịch cổ/thân ±2°.
- Cửa sổ 2 s (20 khung), bước 0,5 s. **Bỏ** cửa sổ: thiếu điểm / hông khó thấy, nhãn lẫn (BAD 20–80%), > 40% khung sát ngưỡng.

In [ ]:
# Landmark -> góc -> hiệu chỉnh theo người -> nhãn RULA nhóm B -> lọc -> cửa sổ 2 s cho Transformer.
import sys; sys.path.insert(0, WORK)
import json, importlib
import numpy as np, pandas as pd
import posture_features; importlib.reload(posture_features)
from posture_features import (FEATURES, choose_side, facing_sign, sequence_angles, calibration_baseline,
                              apply_calibration, to_model_input, rula_frame_label)

FPS = 10
WIN, STRIDE = 20, 5            # 2 s, bước 0,5 s
MIN_PURE = 0.8                 # >= 80% khung cùng nhãn mới giữ
MIN_STABLE = 0.6               # >= 60% khung "ổn định" (không sát ngưỡng)
MIN_VALID = 0.8                # >= 80% khung có đủ điểm
MIN_VIS_HIP = 0.3              # trung bình visibility hông

files = sorted(f for f in glob.glob(f"{LM_DIR}/*.npz") if "SHOULDER_TILT" not in f)
meta = []
for f in files:
    name = os.path.basename(f)[:-4]
    m = re.match(r"(P\d+)_(S\d+)_(.+)", name)
    z = np.load(f)
    setup = "x".join(str(int(v)) for v in z["wh"]) if "wh" in z.files else "na"
    meta.append(dict(file=f, name=name, person=m.group(1), session=m.group(2), clip=m.group(3), setup=setup))
meta = pd.DataFrame(meta)

X, Y, G, INFO, frames_rows = [], [], [], [], []
for (person, session, setup), grp in meta.groupby(["person", "session", "setup"]):
    data = {r.name: np.load(r.file) for r in grp.itertuples()}
    all_lm = np.concatenate([d["lm"] for d in data.values()])
    all_w = np.concatenate([d["world"] for d in data.values()])
    side = choose_side(all_lm, all_w)
    f = facing_sign(all_lm, side)
    raw = {n: sequence_angles(d["lm"], side, f) for n, d in data.items()}
    calib = [n for n in raw if "CALIBRATION" in n]
    if calib:
        base = calibration_baseline(np.concatenate([raw[n] for n in calib])); src = "CALIBRATION"
    else:
        good = [raw[n][:10 * FPS] for n in raw if "_GOOD_" in n]
        if not good:
            print(f"{person}_{session} [{setup}]: không có CALIBRATION/GOOD -> bỏ"); continue
        base = calibration_baseline(np.concatenate(good)); src = "GOOD (10 s đầu)"
    print(f"{person}_{session} [{setup}, {len(grp)} clip]: side={'trái' if side == 0 else 'phải'} facing={f:+.0f} baseline({src}) "
          + " ".join(f"{k}={v:.1f}" for k, v in base.items()))
    for n, r in raw.items():
        if "CALIBRATION" in n:
            continue
        cal = apply_calibration(r, base)
        labs = [rula_frame_label(row) for row in cal]
        lab = np.array([l for l, _ in labs], float); stab = np.array([s for _, s in labs], bool)
        valid = np.isfinite(cal[:, :2]).all(1)              # cần cổ + thân
        xin = to_model_input(cal)
        clip = n.split("_", 2)[2]
        for t in range(len(cal)):
            frames_rows.append(dict(person=person, session=session, clip=clip, video=n, t=t / FPS,
                                    **{k: cal[t, i] for i, k in enumerate(FEATURES)}, label=lab[t], stable=stab[t]))
        for s in range(0, len(cal) - WIN + 1, STRIDE):
            sl = slice(s, s + WIN)
            v = valid[sl].mean(); l = lab[sl]; ok = np.isfinite(l)
            info = dict(person=person, session=session, clip=clip, video=n, start=s / FPS)
            if v < MIN_VALID or ok.sum() == 0 or np.nanmean(cal[sl, FEATURES.index("vis_hip")]) < MIN_VIS_HIP:
                INFO.append({**info, "kept": False, "reason": "thiếu điểm"}); continue
            pbad = np.nanmean(l)
            if MIN_PURE > pbad > 1 - MIN_PURE:
                INFO.append({**info, "kept": False, "reason": "nhãn lẫn"}); continue
            if stab[sl][ok].mean() < MIN_STABLE:
                INFO.append({**info, "kept": False, "reason": "sát ngưỡng"}); continue
            y = int(pbad >= 0.5)
            X.append(xin[sl]); Y.append(y); G.append(person)
            INFO.append({**info, "kept": True, "reason": "", "label": y})

X, Y, G = np.stack(X), np.array(Y), np.array(G)
np.savez_compressed("dataset_windows.npz", X=X, Y=Y, G=G)
info = pd.DataFrame(INFO); info.to_csv("windows_info.csv", index=False)
pd.DataFrame(frames_rows).to_csv("frames_labeled.csv", index=False)
print("\nCửa sổ giữ lại:", len(Y), "| BAD:", int(Y.sum()), f"({Y.mean():.0%})")
print(info.assign(reason=info.reason.replace("", "giữ lại")).groupby("reason").size().rename("số cửa sổ").to_string())
k = info[info.kept]
print("\n% BAD theo kịch bản quay (kiểm tra nhanh teacher RULA nhóm B):")
print(k.assign(clip=k["clip"].str.replace(r"_\d+$", "", regex=True)).groupby("clip").label.agg(["mean", "size"]).round(2).to_string())
print("\nTheo người:", k.groupby("person").label.agg(["mean", "size"]).round(2).to_dict("index"))

## Phần 5 — Train Transformer + baseline Random Forest + xuất ONNX
- **Đánh giá LOSO** (Leave-One-Subject-Out): train trên các người còn lại, thử trên 1 người. Báo cáo accuracy, Cohen's κ, độ nhạy BAD, độ đặc hiệu GOOD, F1 lớp BAD.
- **Baseline Random Forest** trên thống kê cửa sổ (mean/std/min/max) để chứng minh Transformer có giá trị.
- Model cuối train trên **toàn bộ** dữ liệu → `.pt` + `.onnx` + `posture_config.json`.

In [ ]:
import json, numpy as np, torch, torch.nn as nn
from sklearn.ensemble import RandomForestClassifier
from posture_features import FEATURES, CALIBRATED, ANGLE_SCALE, SHOULDER_TILT_ON, SHOULDER_TILT_OFF, MIN_SHOULDER_VIS

torch.manual_seed(0); np.random.seed(0)
d = np.load("dataset_windows.npz", allow_pickle=True)
X, Y, G = d["X"], d["Y"].astype(np.float32), d["G"]
T, F = X.shape[1:]
ANG_IDX = [i for i, k in enumerate(FEATURES) if not k.startswith("vis_")]
CAL_IDX = [FEATURES.index(k) for k in CALIBRATED]


class PostureTransformer(nn.Module):
    def __init__(self, f=F, t=T, d=64, heads=4, layers=2, ff=128, drop=0.1):
        super().__init__()
        self.inp = nn.Linear(f, d)
        self.pos = nn.Parameter(torch.zeros(1, t, d))
        enc = nn.TransformerEncoderLayer(d, heads, ff, drop, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(enc, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, 1)

    def forward(self, x):                       # x: (B,T,F) -> logit (B,)
        h = self.enc(self.inp(x) + self.pos)
        return self.head(self.norm(h.mean(1))).squeeze(-1)


def augment(xb):
    # Cam chạy thật cũng đặt 90° -> chỉ mô phỏng sai số đặt cam / hiệu chỉnh / nhiễu MediaPipe.
    xb = xb.clone(); B = xb.shape[0]
    xb[:, :, ANG_IDX] *= torch.empty(B, 1, 1).uniform_(0.95, 1.05)                    # đặt cam lệch vài độ
    xb[:, :, CAL_IDX] += torch.randn(B, 1, len(CAL_IDX)) * (2.0 / ANGLE_SCALE)         # lệch hiệu chỉnh ±2°
    xb[:, :, ANG_IDX] += torch.randn(B, T, len(ANG_IDX)) * (1.0 / ANGLE_SCALE)         # rung từng khung
    drop = torch.rand(B, T, 1) < 0.05                                                  # mất khung
    return torch.where(drop, torch.zeros_like(xb), xb)


def fit(Xtr, Ytr, epochs=40, lr=2e-3, bs=128, verbose=False):
    m = PostureTransformer(); opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=epochs * (len(Xtr) // bs + 1))
    pos_w = torch.tensor((1 - Ytr.mean()) / max(Ytr.mean(), 1e-3))
    lossf = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    Xt, Yt = torch.tensor(Xtr), torch.tensor(Ytr)
    for ep in range(epochs):
        m.train(); perm = torch.randperm(len(Xt)); tot = 0
        for i in range(0, len(Xt), bs):
            idx = perm[i:i + bs]
            loss = lossf(m(augment(Xt[idx])), Yt[idx])
            opt.zero_grad(); loss.backward(); opt.step(); sched.step(); tot += loss.item() * len(idx)
        if verbose and (ep % 10 == 9 or ep == epochs - 1):
            print(f"  epoch {ep+1}: loss {tot/len(Xt):.3f}")
    return m.eval()


def scores(y, yh):
    y, yh = y.astype(int), yh.astype(int)
    tp = ((y == 1) & (yh == 1)).sum(); tn = ((y == 0) & (yh == 0)).sum()
    fp = ((y == 0) & (yh == 1)).sum(); fn = ((y == 1) & (yh == 0)).sum()
    acc = (tp + tn) / len(y)
    pe = (y == 1).mean() * (yh == 1).mean() + (y == 0).mean() * (yh == 0).mean()
    return dict(n=len(y), acc=acc, kappa=(acc - pe) / (1 - pe + 1e-9), sens_bad=tp / max(tp + fn, 1),
                spec_good=tn / max(tn + fp, 1), f1_bad=2 * tp / max(2 * tp + fp + fn, 1))


def predict(m, Xte):
    with torch.no_grad():
        return (torch.sigmoid(m(torch.tensor(Xte))).numpy() >= 0.5).astype(int)


def win_stats(Xw):   # baseline RF: mean/std/min/max mỗi đặc trưng trong cửa sổ
    return np.concatenate([Xw.mean(1), Xw.std(1), Xw.min(1), Xw.max(1)], 1)


fmt = lambda r: " ".join(f"{k}={v:.3f}" if isinstance(v, float) else f"{k}={v}" for k, v in r.items())
print(f"Dữ liệu: {len(Y)} cửa sổ, {Y.mean():.0%} BAD, đặc trưng {FEATURES}\n")
res, res_rf = {}, {}
for p in np.unique(G):
    tr, te = G != p, G == p
    res[p] = {k: float(v) for k, v in scores(Y[te], predict(fit(X[tr], Y[tr]), X[te])).items()}
    rf = RandomForestClassifier(300, class_weight="balanced", n_jobs=-1, random_state=0).fit(win_stats(X[tr]), Y[tr])
    res_rf[p] = {k: float(v) for k, v in scores(Y[te], rf.predict(win_stats(X[te]))).items()}
    print(f"LOSO {p} | Transformer: {fmt(res[p])}")
    print(f"LOSO {p} | RandomForest: {fmt(res_rf[p])}")
mean = lambda R: {k: float(np.mean([r[k] for r in R.values()])) for k in ("acc", "kappa", "sens_bad", "spec_good", "f1_bad")}
print("\nTrung bình LOSO | Transformer :", fmt(mean(res)))
print("Trung bình LOSO | RandomForest:", fmt(mean(res_rf)))

print("\nTrain model cuối trên toàn bộ dữ liệu ...")
model = fit(X, Y, verbose=True)
torch.save(model.state_dict(), f"{OUT_DIR}/posture_transformer.pt")
torch.onnx.export(model, torch.zeros(1, T, F), f"{OUT_DIR}/posture_transformer.onnx", input_names=["x"], output_names=["logit"],
                  dynamic_axes={"x": {0: "batch"}, "logit": {0: "batch"}}, opset_version=17, dynamo=False)
cfg = dict(version="v3", teacher="RULA nhóm B + LEAN_BACKWARD", side_cam_deg=90,
           features=FEATURES, calibrated=CALIBRATED, angle_scale=ANGLE_SCALE, window=int(T), fps=10, threshold=0.5,
           shoulder_tilt_on=SHOULDER_TILT_ON, shoulder_tilt_off=SHOULDER_TILT_OFF, min_shoulder_vis=MIN_SHOULDER_VIS,
           n_train_windows=int(len(Y)), bad_ratio=float(Y.mean()),
           loso_transformer=res, loso_random_forest=res_rf)
json.dump(cfg, open(f"{OUT_DIR}/posture_config.json", "w"), indent=2, ensure_ascii=False)
print("Đã lưu posture_transformer.pt / .onnx / posture_config.json")

## Phần 6 — Kiểm tra file ONNX chạy được

In [ ]:
import onnxruntime as ort
s = ort.InferenceSession(f'{OUT_DIR}/posture_transformer.onnx')
idx = np.r_[np.where(Y == 0)[0][:4], np.where(Y == 1)[0][:4]]
xb = X[idx]
p_onnx = 1 / (1 + np.exp(-s.run(None, {'x': xb})[0]))
with torch.no_grad():
    p_torch = torch.sigmoid(model(torch.tensor(xb))).numpy()
print('ONNX:', p_onnx.round(3)); print('Torch:', p_torch.round(3)); print('Nhãn:', Y[idx])
assert np.allclose(p_onnx, p_torch, atol=1e-4)
print('✅ ONNX khớp PyTorch')

## Phần 7 — Kiểm tra công thức lệch vai trên clip cam trước (không train)

`tilt = atan2(dy, dx)` giữa 2 vai (pixel), `tilt_rel = tilt − baseline`, median 5 khung. Lệch khi `|tilt_rel| > 3.5°`, hết lệch khi `< 2.5°`. (Chạy thử trên P01: đoạn nghiêng vai chủ ý chỉ lệch ~4–5°, ngồi thẳng dao động ~±2° → ngưỡng 7° ban đầu bỏ sót. Chỉnh `SHOULDER_TILT_ON/OFF` trong Phần 2 nếu cần.)

Clip SHOULDER_TILT hiện **không có clip CALIBRATION cam trước**, nên baseline tạm lấy trung vị **`BASE_SEC` giây đầu** (giả định lúc bắt đầu quay người ngồi thẳng). Nếu không đúng, sửa `BASE_SEC` hoặc quay thêm 10 s ngồi chuẩn bằng cam trước. Đây chỉ là kiểm tra nhanh; để báo cáo độ chính xác, cần đánh dấu tay đoạn nào đang nghiêng.

In [ ]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from posture_features import sequence_shoulder_tilt, SHOULDER_TILT_ON, SHOULDER_TILT_OFF
BASE_SEC = 2.0
front = sorted(glob.glob(f"{LM_DIR}/*SHOULDER_TILT*.npz"))
if not front:
    print("Không có clip cam trước (SHOULDER_TILT) — bỏ qua phần này.")
else:
    fig, axes = plt.subplots(len(front), 1, figsize=(11, 3 * len(front)), squeeze=False)
    for ax, fpath in zip(axes[:, 0], front):
        z = np.load(fpath); t = z["t"] - z["t"][0]
        tilt = sequence_shoulder_tilt(z["lm"])
        base = np.nanmedian(tilt[t < BASE_SEC])
        rel = pd.Series(tilt - base).rolling(5, center=True, min_periods=1).median().to_numpy()
        ok = np.isfinite(rel)
        print(f"{os.path.basename(fpath)[:-4]}: {ok.mean():.0%} khung đo được | baseline {base:+.1f}° | "
              f"max |lệch| {np.nanmax(np.abs(rel)):.1f}° | khung lệch > {SHOULDER_TILT_ON:.0f}°: {(np.abs(rel[ok]) > SHOULDER_TILT_ON).mean():.0%}")
        ax.plot(t, rel, lw=1.5, label="lệch vai so với baseline (°)")
        for y in (SHOULDER_TILT_ON, -SHOULDER_TILT_ON):
            ax.axhline(y, ls="--", c="r", lw=1)
        for y in (SHOULDER_TILT_OFF, -SHOULDER_TILT_OFF):
            ax.axhline(y, ls=":", c="gray", lw=1)
        ax.set_title(os.path.basename(fpath)[:-4]); ax.set_xlabel("giây"); ax.set_ylabel("độ"); ax.legend(loc="upper right")
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/shoulder_tilt_check.png", dpi=110); plt.show()

## Phần 8 — Script chạy realtime (2 cam: hông 90° + laptop chính diện)

In [ ]:
%%writefile /kaggle/working/posture_model/realtime_posture.py
"""
Chạy realtime với 2 camera:
  - Cam hông (đặt ~90°, ngang vai)  -> MediaPipe Pose -> góc cổ/thân -> Transformer (ONNX) -> GOOD/BAD
  - Cam laptop (chính diện)         -> MediaPipe Pose -> công thức lệch vai (không train) -> SHOULDER_TILT
  -> median filter + debounce (+ cooldown) cho từng nhánh -> cảnh báo.

  python realtime_posture.py --side 1 --front 0                 # cam hông = webcam 1, cam laptop = webcam 0
  python realtime_posture.py --side 0 --front -1                # chỉ cam hông (tắt đo lệch vai)
  python realtime_posture.py --side-video hong.mp4 --front-video truoc.mp4   # chạy thử trên 2 video quay cùng lúc

Phím: q = thoát, c = hiệu chỉnh lại (ngồi thẳng, nhìn màn hình 10 giây).
"""
import argparse, collections, json, os, time, urllib.request
import cv2, numpy as np, onnxruntime as ort
from posture_features import (FEATURES, IDX, choose_side, facing_sign, sequence_angles, calibration_baseline,
                              apply_calibration, to_model_input, rula_frame_label, shoulder_tilt)

HERE = os.path.dirname(os.path.abspath(__file__))
POSE_URL = ("https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
            "pose_landmarker_full/float16/latest/pose_landmarker_full.task")


# ------------------------------------------------------------------ MediaPipe (Tasks API, dự phòng Solutions)
class PoseDetector:
    def __init__(self, task_path):
        import mediapipe as mp
        self.mp = mp
        try:
            from mediapipe.tasks import python as mpt
            from mediapipe.tasks.python import vision
            if not os.path.exists(task_path):
                print("Đang tải pose_landmarker_full.task ..."); urllib.request.urlretrieve(POSE_URL, task_path)
            opts = vision.PoseLandmarkerOptions(base_options=mpt.BaseOptions(model_asset_path=task_path),
                                                running_mode=vision.RunningMode.VIDEO, num_poses=1)
            self.det = vision.PoseLandmarker.create_from_options(opts); self.mode = "tasks"
        except Exception as e:
            print("Tasks API lỗi (", e, ") -> dùng mp.solutions.pose")
            self.det = mp.solutions.pose.Pose(model_complexity=1, smooth_landmarks=True); self.mode = "solutions"
        self.last_ts = -1

    def __call__(self, frame_bgr, ts_ms):
        h, w = frame_bgr.shape[:2]
        rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        if self.mode == "tasks":
            ts_ms = max(ts_ms, self.last_ts + 1); self.last_ts = ts_ms      # timestamp phải tăng dần
            r = self.det.detect_for_video(self.mp.Image(image_format=self.mp.ImageFormat.SRGB, data=rgb), ts_ms)
            if not r.pose_landmarks:
                return None, None
            lm, wl = r.pose_landmarks[0], r.pose_world_landmarks[0]
        else:
            r = self.det.process(rgb)
            if not r.pose_landmarks:
                return None, None
            lm, wl = r.pose_landmarks.landmark, r.pose_world_landmarks.landmark
        L = np.array([[p.x * w, p.y * h, getattr(p, "visibility", 1.0) or 0.0] for p in lm], np.float32)
        W = np.array([[p.x, p.y, p.z] for p in wl], np.float32)
        return L, W


# ------------------------------------------------------------------ debounce + cooldown
class Debounce:
    """bad=True/False/None (None = không rõ, giữ nguyên). BAD liên tục >= alert_sec -> CANH BAO;
    GOOD liên tục >= clear_sec -> GOOD. Sự kiện cảnh báo (in/bíp/gửi server) cách nhau >= cooldown."""

    def __init__(self, alert_sec, clear_sec, cooldown):
        self.alert_sec, self.clear_sec, self.cooldown = alert_sec, clear_sec, cooldown
        self.reset()

    def reset(self):
        self.status, self.bad_since, self.good_since, self.last_event = "...", None, None, -1e9

    def update(self, bad, now):
        event = False
        if bad is True:
            self.good_since = None
            self.bad_since = now if self.bad_since is None else self.bad_since
            if now - self.bad_since >= self.alert_sec:
                if self.status != "CANH BAO" and now - self.last_event >= self.cooldown:
                    event, self.last_event = True, now
                self.status = "CANH BAO"
            elif self.status != "CANH BAO":
                self.status = "BAD"
        elif bad is False:
            self.bad_since = None
            self.good_since = now if self.good_since is None else self.good_since
            if now - self.good_since >= self.clear_sec:
                self.status = "GOOD"
        return event


COL = {"GOOD": (60, 200, 60), "BAD": (0, 165, 255), "CANH BAO": (0, 0, 255), "...": (180, 180, 180)}


def draw_side(frame, L, side, status, prob, ang, rula_lab, calib_left=None):
    if L is not None and side is not None:
        pts = {k: tuple(int(v) for v in L[IDX[k][side], :2]) for k in IDX}
        for a, b in [("ear", "shoulder"), ("shoulder", "hip"), ("ear", "eye")]:
            cv2.line(frame, pts[a], pts[b], (255, 255, 255), 3)
        for p in pts.values():
            cv2.circle(frame, p, 6, (0, 200, 255), -1)
    h, w = frame.shape[:2]
    cv2.rectangle(frame, (0, 0), (w, 90), (30, 30, 30), -1)
    cv2.putText(frame, "CAM HONG - co / than", (15, 25), 0, 0.7, (200, 200, 200), 2)
    if calib_left is not None:
        cv2.putText(frame, f"HIEU CHINH: ngoi thang... {calib_left:.0f}s", (15, 70), 0, 1.0, (0, 255, 255), 2)
        return frame
    cv2.putText(frame, status, (15, 75), 0, 1.4, COL[status], 4)
    txt = f"P(BAD)={prob:.2f}"
    if ang is not None:
        txt += f" | co {ang['neck']:+.0f} than {ang['trunk']:+.0f} dau {ang['head_pitch']:+.0f}"
    if rula_lab is not None and np.isfinite(rula_lab):
        txt += f" | RULA-B: {'BAD' if rula_lab else 'GOOD'}"
    cv2.putText(frame, txt, (260, 70), 0, 0.65, (255, 255, 255), 2)
    if status == "CANH BAO":
        cv2.rectangle(frame, (0, 0), (w - 1, h - 1), (0, 0, 255), 12)
    return frame


def draw_front(frame, L, status, tilt_rel, calib_left=None):
    if L is not None:
        a, b = tuple(int(v) for v in L[11, :2]), tuple(int(v) for v in L[12, :2])
        cv2.line(frame, a, b, (255, 255, 255), 3)
        for p in (a, b):
            cv2.circle(frame, p, 7, (0, 200, 255), -1)
    h, w = frame.shape[:2]
    cv2.rectangle(frame, (0, 0), (w, 90), (30, 30, 30), -1)
    cv2.putText(frame, "CAM LAPTOP - lech vai", (15, 25), 0, 0.7, (200, 200, 200), 2)
    if calib_left is not None:
        cv2.putText(frame, f"HIEU CHINH... {calib_left:.0f}s", (15, 70), 0, 1.0, (0, 255, 255), 2)
        return frame
    label = {"GOOD": "VAI CAN", "BAD": "LECH VAI", "CANH BAO": "LECH VAI!", "...": "..."}[status]
    cv2.putText(frame, label, (15, 75), 0, 1.4, COL[status], 4)
    if tilt_rel is not None and np.isfinite(tilt_rel):
        cv2.putText(frame, f"lech {tilt_rel:+.1f} do", (330, 70), 0, 0.8, (255, 255, 255), 2)
    if status == "CANH BAO":
        cv2.rectangle(frame, (0, 0), (w - 1, h - 1), (0, 0, 255), 12)
    return frame


def hstack(a, b, height=540):
    rs = lambda f: cv2.resize(f, (int(f.shape[1] * height / f.shape[0]), height))
    return np.hstack([rs(a), rs(b)]) if b is not None else rs(a)


# ------------------------------------------------------------------ chính
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--side", type=int, default=1, help="chỉ số webcam đặt bên hông (~90°)")
    ap.add_argument("--front", type=int, default=0, help="chỉ số webcam laptop chính diện, -1 = tắt")
    ap.add_argument("--side-video", default=None)
    ap.add_argument("--front-video", default=None)
    ap.add_argument("--calib-sec", type=float, default=10)
    ap.add_argument("--alert-sec", type=float, default=5, help="BAD kéo dài bao lâu mới cảnh báo")
    ap.add_argument("--clear-sec", type=float, default=3, help="GOOD kéo dài bao lâu mới tắt cảnh báo")
    ap.add_argument("--cooldown", type=float, default=60, help="khoảng cách tối thiểu giữa 2 cảnh báo cùng loại")
    ap.add_argument("--log", default=None, help="ghi CSV kết quả")
    a = ap.parse_args()

    cfg = json.load(open(os.path.join(HERE, "posture_config.json")))
    sess = ort.InferenceSession(os.path.join(HERE, "posture_transformer.onnx"), providers=["CPUExecutionProvider"])
    WIN, FPS, THR = cfg["window"], cfg["fps"], cfg["threshold"]
    T_ON, T_OFF = cfg["shoulder_tilt_on"], cfg["shoulder_tilt_off"]
    task = os.path.join(HERE, "pose_landmarker_full.task")

    use_video = a.side_video is not None
    cap_s = cv2.VideoCapture(a.side_video if use_video else a.side)
    if not cap_s.isOpened():
        raise SystemExit("Không mở được cam hông")
    cap_f = None
    if a.front_video or (not use_video and a.front >= 0):
        cap_f = cv2.VideoCapture(a.front_video if a.front_video else a.front)
        if not cap_f.isOpened():
            print("Không mở được cam laptop -> chỉ chạy nhánh cổ/thân"); cap_f = None
    det_s = PoseDetector(task)
    det_f = PoseDetector(task) if cap_f is not None else None
    vid_fps = cap_s.get(cv2.CAP_PROP_FPS) or 30

    logf = open(a.log, "w", encoding="utf-8") if a.log else None
    if logf:
        logf.write("time,posture,p_bad," + ",".join(FEATURES) + ",rula_b,shoulder,tilt_rel\n")

    deb_p = Debounce(a.alert_sec, a.clear_sec, a.cooldown)          # nhánh cổ/thân
    deb_t = Debounce(a.alert_sec, a.clear_sec, a.cooldown)          # nhánh lệch vai

    def start_calib():
        deb_p.reset(); deb_t.reset()
        return dict(L=[], W=[], tilt=[], t0=None)

    cal, side, f, base, tilt_base = start_calib(), None, None, None, None
    buf = collections.deque(maxlen=WIN)
    p_hist, t_hist = collections.deque(maxlen=5), collections.deque(maxlen=5)      # median filter 5 kết quả
    p_med, t_med, ang, rula_lab = 0.0, np.nan, None, None
    last_s = last_f = -1e9
    Ls_draw = Lf_draw = None
    frame_i, t_start = 0, time.time()

    while True:
        ok, fs = cap_s.read()
        if not ok:
            break
        ff = None
        if cap_f is not None:
            okf, ff = cap_f.read()
            if not okf:
                ff = None
        frame_i += 1
        now = (frame_i / vid_fps) if use_video else (time.time() - t_start)
        ts = int(now * 1000)
        sample_s = now - last_s >= 1.0 / FPS                      # lấy mẫu ~10 fps như lúc train
        sample_f = ff is not None and now - last_f >= 1.0 / FPS
        Ls, Ws = det_s(fs, ts) if sample_s else (None, None)
        Lf = det_f(ff, ts)[0] if sample_f else None
        if sample_s: last_s, Ls_draw = now, Ls                   # giữ khung xương giữa 2 lần lấy mẫu để vẽ
        if sample_f: last_f, Lf_draw = now, Lf

        if base is None:                                          # ---------- hiệu chỉnh
            if cal["t0"] is None:
                cal["t0"] = now
            if Ls is not None:
                cal["L"].append(Ls); cal["W"].append(Ws)
            if Lf is not None:
                cal["tilt"].append(shoulder_tilt(Lf))
            left = a.calib_sec - (now - cal["t0"])
            if left <= 0:
                if len(cal["L"]) < 10:
                    print("Cam hông không thấy người đủ lâu, hiệu chỉnh lại."); cal = start_calib()
                else:
                    Lc, Wc = np.stack(cal["L"]), np.stack(cal["W"])
                    side = choose_side(Lc, Wc); f = facing_sign(Lc, side)
                    base = calibration_baseline(sequence_angles(Lc, side, f))
                    tilts = np.array(cal["tilt"], float)
                    tilt_base = float(np.nanmedian(tilts)) if np.isfinite(tilts).sum() >= 5 else None
                    print(f"Hiệu chỉnh xong: phía {'trái' if side == 0 else 'phải'}, baseline",
                          {k: round(v, 1) for k, v in base.items()},
                          "| lệch vai baseline:", None if tilt_base is None else round(tilt_base, 1))
                    buf.clear(); p_hist.clear(); t_hist.clear()
            lc = max(left, 0)
            vis_s = draw_side(fs, Ls_draw, side, "...", 0, None, None, calib_left=lc)
            vis_f = draw_front(ff, Lf_draw, "...", None, calib_left=lc) if ff is not None else None
        else:                                                     # ---------- theo dõi
            now_s = time.strftime("%H:%M:%S")
            # nhánh 1: cổ / thân (student)
            if sample_s:
                if Ls is not None:
                    c = apply_calibration(sequence_angles(Ls[None], side, f), base)
                    ang = {k: float(c[0, i]) for i, k in enumerate(FEATURES)}
                    rula_lab, _ = rula_frame_label(c[0])
                    buf.append(to_model_input(c)[0])
                else:
                    ang = rula_lab = None
                    buf.append(np.zeros(len(FEATURES), np.float32))
                if len(buf) == WIN:
                    logit = sess.run(None, {"x": np.stack(buf)[None]})[0][0]
                    p_hist.append(1 / (1 + np.exp(-logit)))
                    p_med = float(np.median(p_hist))
                    bad = True if p_med >= THR + 0.1 else False if p_med <= THR - 0.1 else None
                    if deb_p.update(bad, now):
                        print(f"[{now_s}] CẢNH BÁO tư thế cổ/thân (P(BAD)={p_med:.2f})")
            # nhánh 2: lệch vai (công thức)
            tilt_rel = np.nan
            if sample_f and tilt_base is not None:
                tv = shoulder_tilt(Lf) if Lf is not None else np.nan
                if np.isfinite(tv):
                    t_hist.append(tv - tilt_base)
                if t_hist:
                    t_med = float(np.median(t_hist))
                    bad = True if abs(t_med) > T_ON else False if abs(t_med) < T_OFF else None
                    if deb_t.update(bad if np.isfinite(tv) else None, now):
                        print(f"[{now_s}] CẢNH BÁO lệch vai ({t_med:+.1f}°)")
            if logf and sample_s:
                logf.write(f"{now:.2f},{deb_p.status},{p_med:.3f},"
                           + ",".join(f"{ang[k]:.2f}" if ang else "" for k in FEATURES)
                           + f",{rula_lab},{deb_t.status if tilt_base is not None else ''},{t_med:.2f}\n")
            vis_s = draw_side(fs, Ls_draw, side, deb_p.status, p_med, ang, rula_lab)
            vis_f = draw_front(ff, Lf_draw, deb_t.status if tilt_base is not None else "...", t_med) if ff is not None else None

        cv2.imshow("Posture monitor (q: thoat, c: hieu chinh lai)", hstack(vis_s, vis_f))
        k = cv2.waitKey(1) & 0xFF
        if k == ord("q"):
            break
        if k == ord("c"):
            cal, base, tilt_base = start_calib(), None, None
    cap_s.release()
    if cap_f is not None:
        cap_f.release()
    cv2.destroyAllWindows()
    if logf:
        logf.close()


if __name__ == "__main__":
    main()

## Phần 9 — Đóng gói để tải về

In [ ]:
import shutil
shutil.copy(f'{WORK}/posture_features.py', OUT_DIR)
shutil.copy(TASK, OUT_DIR)
open(f'{OUT_DIR}/requirements.txt', 'w').write('mediapipe\nopencv-python\nonnxruntime\nnumpy\n')
for f in ['windows_info.csv', 'frames_labeled.csv']:
    if os.path.exists(f'{WORK}/{f}'): shutil.copy(f'{WORK}/{f}', OUT_DIR)
shutil.make_archive(f'{WORK}/posture_model', 'zip', OUT_DIR)
print(sorted(os.listdir(OUT_DIR)))
print('✅ Tải về: /kaggle/working/posture_model.zip')